# 1. 环境配置

## 1.1 python 环境准备

In [1]:
! pip install gradio==6.2.0 openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 beautifulso|up4==4.14.3 langchain_chroma==1.1.0

'up4' �����ڲ����ⲿ���Ҳ���ǿ����еĳ���
���������ļ���


## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [2]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

## 1.3 代码准备

由于存储到向量数据库前需要先准备好文档以及切分好的文档块，因此这里需要把上一节的内容进行载入：

In [3]:
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://zh.d2l.ai/chapter_introduction/index.html")
docs = loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
 chunk_size = 1500,
 chunk_overlap = 150)
splits = text_splitter.split_documents(docs)
print(len(splits))

USER_AGENT environment variable not set, consider setting it to identify your requests.


27


# 2. 向量数据库生成

## 2.1 简介

在将每一个文档切割成合适的 chunk 后，我们还需要进行文本嵌入及向量数据库存储：
- 文本嵌入：使用 Embedding 模型将文本转换成高维向量（如 1536 维）。
- 向量存储：将向量和元数据（metadata）一起存入向量数据库中。

存储完后，就相当于给文本建一个‘语义索引’，让模型能按‘意思相近’而不是‘关键词相同’来查资料。

## 2.2 Embedding 模型

Embeddings 的本质是将一段文本转化为一长串的向量，这些向量实际上是对文字的一种数字化表示。假如两段文本内容越相关，其在向量空间中的距离也是越近的。因此我们可以通过这个特性来检索到最相关的片段内容。

许多公司都有推出自己的 embedding 模型：
- OpenAI（text-embedding-3-large）
- 阿里云（DashScopeEmbedding）
- Google（Gemini text-embedding-004）
- huggingface 上的开源模型（如 BERT、E5 等）

In [4]:
from langchain_community.embeddings import DashScopeEmbeddings
import os

# 设置 embedding 模型（阿里云）
embeddings = DashScopeEmbeddings(
  dashscope_api_key=os.getenv('DASHSCOPE_API_KEY'))

# 设置文本内容
text_1 = "今天天气不错"

# 进行文本向量化
query_result = embeddings.embed_query(text_1)
print(query_result)

[-3.6761717796325684, 3.9287760257720947, 1.406152367591858, 2.5160155296325684, -1.5029622316360474, -2.227083444595337, 1.3093180656433105, -0.249114990234375, 0.09683430939912796, 5.202343940734863, -2.157926321029663, 2.1783854961395264, 1.2405558824539185, 1.2792236804962158, -1.8366210460662842, 3.848893165588379, -1.5490397214889526, 7.020751953125, 0.12340494990348816, 0.6939046382904053, -6.4620442390441895, -1.3764973878860474, 3.9332518577575684, -2.6236329078674316, 1.41455078125, -0.0037109374534338713, 2.8648884296417236, -0.48908692598342896, -1.6942057609558105, -2.1673176288604736, -1.2521158456802368, 0.13245442509651184, 4.077538967132568, -0.6785807013511658, 2.2517008781433105, 4.3818359375, 1.63214111328125, 2.691723585128784, -0.20652669668197632, 0.38512369990348816, 0.6812215447425842, 2.8853495121002197, -3.4527180194854736, -4.7725911140441895, 3.174837350845337, 0.3503173887729645, 1.2563313245773315, -0.782818615436554, -0.3004313111305237, -1.5382486581802

## 2.3 向量数据库存储

在准备好了 embedding 模型后，我们还需要解决的一个问题是用哪一个向量数据库进行向量的存储。

### 2.3.1 InMemoryVectorStore
在 LangChain 中最基础的向量数据库就是基于内存的 InMemoryVectorStore 。

In [5]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.embeddings import DashScopeEmbeddings
import os

vector_store = InMemoryVectorStore(embedding=DashScopeEmbeddings(
 dashscope_api_key=os.getenv('DASHSCOPE_API_KEY')))

其主要包含三个方法（其他向量数据库也类似）：
- 添加文档：vector_store.add_documents(documents=[doc1], ids=["id1"])
- 删除文档：vector_store.delete(ids=["id1"])
- 相似度检索：vector_store.similarity_search("your query here")

In [6]:
vectordb = vector_store.from_documents(
  documents=splits,
  embedding=embeddings)

print(len(vectordb.store))

27


### 2.3.2 Chroma

但是 InMemoryVectorStore 无法进行长期保存，当程序运行结束后，向量数据库内的内容将自动清除。

因此为了能够更长久的保存，Langchain 提供了更专业的向量数据库支持，包括 Chroma、Pinecone 以及 FAISS 等。

那这里我就以 Chroma 为例来展示一下具体的使用方式（先安装相关库）：

In [7]:
from langchain_community.embeddings import DashScopeEmbeddings
from langchain_chroma import Chroma
import os

embeddings = DashScopeEmbeddings(
  dashscope_api_key=os.getenv('DASHSCOPE_API_KEY'))

vectordb = Chroma.from_documents(
  documents=splits,
  embedding=embeddings,
  persist_directory='./chroma')

print(vectordb._collection.count())

27
